# Kanitakorn TH-SFT — Kaggle T4×2 baseline + pipeline validation

Run on Kaggle with a **T4×2 (free)** or **TPU v3-8** runtime. Goal: validate the
full inference + format-extractor + judge pipeline produces correct
deltas-vs-baseline using a real (small) open-weights model, BEFORE renting
the A100 for the 35B SFT run.

**Model under test**: `Qwen/Qwen2.5-7B-Instruct` (fits in T4 16GB with FP16)
**Eval subset**: 100 records from `dataset/reports/benchmark_inputs.jsonl`
(stratified ~12 per family)

Before running:
1. Upload `dataset/reports/benchmark_inputs.jsonl` as a Kaggle dataset
2. Upload the `tools/` directory as a Kaggle dataset
3. Enable GPU runtime (Settings → Accelerator → GPU T4×2)
4. Run all cells

In [ ]:
# Cell 1: install deps
!pip install -q transformers accelerate sentencepiece sympy jsonschema pythainlp 2>&1 | tail -3

In [ ]:
# Cell 2: import + setup
import sys, os, json, time
from pathlib import Path

# Adjust if your Kaggle dataset names differ.
TOOLS_DIR = Path('/kaggle/input/kanitakorn-tools/tools')  # rename your tools-input here
INPUTS_PATH = Path('/kaggle/input/kanitakorn-benchmark-inputs/benchmark_inputs.jsonl')
OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(TOOLS_DIR))
import torch
print('CUDA:', torch.cuda.is_available(), '— device count:', torch.cuda.device_count())

In [ ]:
# Cell 3: load model (Qwen2.5-7B-Instruct fits in T4×2 with device_map='auto')
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
print(f'loaded {MODEL_ID} in {time.time()-t0:.1f}s')

In [ ]:
# Cell 4: pick a stratified subset of 100 inputs (~12 per family)
from collections import defaultdict
import random
rng = random.Random(42)
by_family = defaultdict(list)
for line in INPUTS_PATH.read_text(encoding='utf-8').splitlines():
    if not line.strip(): continue
    r = json.loads(line)
    by_family[r['family']].append(r)

PER_FAMILY = 12  # adjust as you like
subset = []
for fam, recs in by_family.items():
    if fam == 'hotpotqa':
        continue  # 7405 records; skip for the smoke run
    rng.shuffle(recs)
    subset.extend(recs[:PER_FAMILY])
print(f'subset size: {len(subset)}')

In [ ]:
# Cell 5: family-specific system prompts (mirrors tools/run_inference.py)
FAMILY_SYS = {
    'aime24': 'You are a math expert. Solve this problem. End with \\boxed{ANSWER}.',
    'aime25': 'You are a math expert. Solve this problem. End with \\boxed{ANSWER}.',
    'math500': 'You are a math expert. Solve this problem. End with \\boxed{ANSWER}.',
    'ifeval': "Follow the user's instruction precisely. Output ONLY the response.",
    'mt_bench': 'You are a helpful assistant. Answer in Thai if the question is in Thai.',
    'openthaieval': 'You are a Thai academic-exam expert. Answer with (1), (2), (3), or (4).',
    'livecodebench': 'You are a competitive programmer. Output ONLY Python code in ```python ... ```.',
}

def generate(prompt, system='', max_new=400):
    msgs = [{'role':'system','content':system}] if system else []
    msgs.append({'role':'user','content':prompt[:1500]})
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
# Cell 6: run inference on the subset
preds_path = OUT_DIR / 'qwen2.5-7b_predictions.jsonl'
t0 = time.time()
with preds_path.open('w', encoding='utf-8') as fh:
    for i, rec in enumerate(subset, start=1):
        sys_p = FAMILY_SYS.get(rec['family'], '')
        try:
            rec['prediction'] = generate(rec['prompt'], system=sys_p)
        except Exception as e:
            rec['prediction'] = ''
            rec['error'] = f'{type(e).__name__}: {e}'
        # IFEval needs a constraints list; here we just punt with a regex-match-all to score deterministic path.
        if rec['family'] == 'ifeval':
            rec.setdefault('constraints', [{'type':'regex','pattern':'.+'}])
        fh.write(json.dumps(rec, ensure_ascii=False) + '\n')
        if i % 10 == 0:
            print(f'  [{i}/{len(subset)}] {time.time()-t0:.0f}s elapsed')
print(f'done. wrote {preds_path}')

In [ ]:
# Cell 7: score predictions through the project's benchmark_eval + judges
from benchmark_eval import score_predictions, write_report, TYPHOON_2_BASELINES
results = score_predictions(preds_path)
report_path = OUT_DIR / 'qwen2.5-7b_report.md'
write_report(results, TYPHOON_2_BASELINES, 'Qwen2.5-7B-Instruct-T4', report_path)
print(open(report_path).read())

## Interpretation

Qwen2.5-7B-Instruct is **non-Thai-tuned** and **smaller than our target** (35B).
Its numbers establish the floor a Thai-SFT'd 35B model must clear by a wide
margin. Typical published Qwen2.5-7B numbers on similar Thai benchmarks fall
in the 0.40–0.55 range; your trained 35B-A3B-Instruct + Kanitakorn SFT should
land 0.10–0.20 above that on every family.

If this notebook completes cleanly and the report renders deltas, the entire
pipeline (extractor → judge → report) is validated — go ahead and rent the A100.